# Rule-Based Information Extraction

## Overview

This notebook presents a rule-based approach for extracting structured information from clinical trial abstracts in the **EBM-NLP dataset**. The objective is to identify and extract the key components of the PIO framework—**Population, Intervention, and Outcome (PIO)**—from unstructured biomedical text.

The extraction pipeline explores multiple rule-based techniques of increasing complexity, including:

- **Unigram-based matching**
- **Combined Unigram + Bigram matching**
- **spaCy-based linguistic processing and pattern matching**

These approaches rely on predefined vocabularies, token patterns, and linguistic rules rather than machine learning models. Rule-based methods provide a transparent and interpretable baseline for evaluating information extraction performance on clinical trial abstracts.


In [1]:
import pandas as pd

In [ ]:
train_df = pd.read_csv("../cwk_train.csv")
test_df  = pd.read_csv("../cwk_test.csv")

In [100]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4457 entries, 0 to 4456
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Unnamed: 0                4457 non-null   int64
 1   Doc_id                    4457 non-null   int64
 2   Documents                 4457 non-null   str  
 3   Tokens                    4457 non-null   str  
 4   Intervention_Lables       4457 non-null   str  
 5   Outcome_Lables            4457 non-null   str  
 6   Participants_Label        4457 non-null   str  
 7   Participants_BIO_Labels   4457 non-null   str  
 8   Interventions_BIO_Lables  4457 non-null   str  
 9   Outcomes_BIO_Lables       4457 non-null   str  
dtypes: int64(2), str(8)
memory usage: 52.4 MB


In [101]:
import ast


In [102]:
cols = train_df.columns.tolist()[3:]

In [103]:

for col in cols:
    train_df[col] = train_df[col].apply(ast.literal_eval)

In [104]:
for col in cols:
    test_df[col] = test_df[col].apply(ast.literal_eval)

In [105]:
participants_bio_labels = train_df["Participants_BIO_Labels"].to_numpy()
interventions_bio_labels = train_df["Interventions_BIO_Lables"].to_numpy()
outcomes_bio_labels = train_df["Outcomes_BIO_Lables"].to_numpy()


In [106]:
participants_bio_labels.shape

(4457,)

In [107]:
def merge_label(p_labels, i_labels, o_labels):
    merged = []

    for p, i, o in zip(p_labels, i_labels, o_labels):
        
        if o != "O":
            tk = o + "-OUT" 
        
        elif i != "O":
            tk = i +"-INT"    

        elif p != "O":
            tk = p + "-PART"

        else:
            tk = "O"

        merged.append(tk)

    return merged

merg_label = []
for i in range(4457):
    ml = merge_label(participants_bio_labels[i], interventions_bio_labels[i], outcomes_bio_labels[i])
    merg_label.append(ml)
    

In [108]:
participants_bio_labels[1]

['O', 'O', 'O', 'O', 'B', 'I', 'I', 'I']

In [109]:
print(merg_label[1])

['B-INT', 'O', 'O', 'O', 'B-PART', 'I-PART', 'I-PART', 'I-PART']


In [110]:
merg_label[1]

['B-INT', 'O', 'O', 'O', 'B-PART', 'I-PART', 'I-PART', 'I-PART']

In [111]:
train_df["Merged BIO Labels"] = merg_label

In [113]:
participants_bio_labels = test_df["Participants_BIO_Labels"].to_numpy()
interventions_bio_labels = test_df["Interventions_BIO_Lables"].to_numpy()
outcomes_bio_labels = test_df["Outcomes_BIO_Lables"].to_numpy()


In [114]:
merg_label = []
for i in range(184):
    ml = merge_label(participants_bio_labels[i], interventions_bio_labels[i], outcomes_bio_labels[i])
    merg_label.append(ml)

In [115]:
test_df["Merged BIO Labels"] = merg_label

In [4]:
#train_df.to_excel("cwk_train_data.xlsx", index=False)
test_df = pd.read_excel(r"Data/cwk_test_data.xlsx")

## Unigram-based matching

In [5]:
import re

PARTICIPANTS = {
    
    "AGE" : re.compile(
    r"\b("
    r"\d+\s*[-–]\s*\d+\s*(years?|yrs?|yr)?|"        
    r"\d+\s*[±\+/-]\s*\d+\s*(years?|yrs?|yr)?|"      
    r"\d+\s*(years?|yrs?|yr)|"                        
    r"\d+|"                                        
    r"adults?|children|infants?|elderly|pediatric|neonates?|adolescents?|teenagers?"  
    r")\b",
    re.IGNORECASE), 
                
    "SEX" : re.compile(
    r"\b("
    r"male|female|men|women|"         
    r"males?|females?|"                                             
    r"boys?|girls?"                     
    r")\b",
    re.IGNORECASE),
    
    "SAMPLE SIZE" : re.compile(
    r"\b("
    r"n\s*=\s*\d+\s*[-–]?\s*\d*|"                        
    r"\d+\s*[-–]?\s*\d*\s+(patients?|subjects?|participants?)|"  
    r"a\s+total\s+of\s+\d+\s*[-–]?\s*\d*\s+(patients?|subjects?|participants?)|" 
    r"\d+\s*[-–]?\s*\d*"                                 
    r")\b",
    re.IGNORECASE)  
    }


INTERVENTION = {
    
    "SURGICAL": re.compile(
    r"\b("
    r"surgical|surgery|surgeries|operative|operation|operations|"
    r"procedure|procedures"
    r")\b",
    re.IGNORECASE),
    
    "PHYSICAL" : re.compile(
    r"\b("
    r"physical|exercise|exercises|training|"
    r"physiotherapy|physical\s+therapy|"
    r"rehabilitation|rehab|"
    r"mobilization|mobilisation"
    r")\b",
    re.IGNORECASE),
    
    "DRUG" : re.compile(
    r"\b("
    r"drug|drugs|"
    r"medication|medications|"
    r"pharmacological|pharmacologic|pharmacotherapy|"
    r"treatment|therapy|"
    r"dose|doses|dosage"
    r")\b",
    re.IGNORECASE),
    
    "EDUCATIONAL" : re.compile(
    r"\b("
    r"education|educational|"
    r"training|"
    r"teaching|"
    r"instruction|"
    r"counseling|counselling|"
    r"awareness|"
    r"program|programme"
    r")\b",
    re.IGNORECASE),
    
    "PSYCHOLOGICAL" : re.compile(
    r"\b("
    r"psychological|"
    r"behavioral|behavioural|"
    r"cognitive|"
    r"therapy|"
    r"counseling|counselling|"
    r"mental|"
    r"psychotherapy"
    r")\b",
    re.IGNORECASE)
}

OUTCOMES = {
    
    "PHYSICAL" : re.compile(
    r"\b("
    r"physical|function|functional|"
    r"mobility|movement|"
    r"recovery|improvement|"
    r"capacity|performance|"
    r"strength|fitness"
    r")\b",
    re.IGNORECASE),
    
    "PAIN" : re.compile(
    r"\b("
    r"pain|"
    r"ache|aches|"
    r"soreness|"
    r"discomfort|"
    r"analgesia"
    r")\b",
    re.IGNORECASE),
    
    "MORTALITY" : re.compile(
    r"\b("
    r"death|deaths|"
    r"mortality|"
    r"fatal|fatality|fatalities|"
    r"survival"
    r")\b",
    re.IGNORECASE),
    
    "ADVERSE_EFFECTS" :re.compile(
    r"\b("
    r"adverse|"
    r"side\s+effect|side\s+effects|"
    r"complication|complications|"
    r"toxicity|toxic|"
    r"harm|"
    r"risk|"
    r"bleeding|infection"
    r")\b",
    re.IGNORECASE),
    
    "MENTAL" : re.compile(
    r"\b("
    r"mental|"
    r"cognitive|"
    r"mood|"
    r"depression|depressive|"
    r"anxiety|anxious|"
    r"stress|stressed|"
    r"psychological|"
    r"well-being|wellbeing|"
    r"emotion|emotional|"
    r"quality\s+of\s+life|QoL"
    r")\b",
    re.IGNORECASE)
}

In [6]:
def token_matching(token):
    token = token.lower().strip()

    for subtype, regex in OUTCOMES.items():
        if regex.search(token):
            return "OUT", subtype
        
    for subtype, regex in INTERVENTION.items():
        if regex.search(token):
            return "INT", subtype
        
    for subtype, regex in PARTICIPANTS.items():
        if regex.search(token):
            return "PART", subtype
        
    return "None", "None"

In [32]:
def entity_labled_tokens(doc_id, tok):
    tokens_list = []
    temp_dict = {"PARTICIPANTS":[], "INTERVENTION":[], "OUTCOMES":[]}
    for i in tok:
        entity, subtype = token_matching(i)
        tokens_list.append(entity)
        
        if entity == "PART":
            temp_dict["PARTICIPANTS"].append(i)  # to be updated if necessary for the subtype
        
        if entity == "INT":
            temp_dict["INTERVENTION"].append(i)
            
        if entity == "OUT":
            temp_dict["OUTCOMES"].append(i)
            
    return doc_id, tokens_list, temp_dict
            

In [ ]:
def BIO_labels(token_list):
    prev = None
    bio_labels = []
    
    for i in token_list:
        if i  == "None":
            label = "O"
            
        elif i == "PART":
            if prev == "PART":
                label = "I" + "-PART"
            else:
                label = "B" + "-PART" 
                
        elif i == "INT":
            if prev == "INT":
                label = "I" + "-INT"
            else:
                label = "B" + "-INT"
                
        elif i == "OUT":
            if prev == "OUT":
                label = "I" + "-OUT"
            else:
                label = "B" + "-OUT"
                
        bio_labels.append(label)
        prev = i
    
    return bio_labels
                       

In [34]:
TOKENS = test_df["Tokens"].to_numpy()
DOC_ID = test_df["Doc_id"].to_numpy()
import ast

MERGED_BIOL = test_df["Merged BIO Labels"].apply(ast.literal_eval).to_numpy()

In [35]:
B_l = []
sub_l = []
for i in range(184):
    doc_id = DOC_ID[i]
    tok = TOKENS[i]
    
    doc_id, tokens_list, temp_dict = entity_labled_tokens(doc_id, tok)
    BIO_Lab = BIO_labels(tokens_list)
    
    B_l.append(BIO_Lab)
    sub_l.append(temp_dict)

In [ ]:
from seqeval.metrics import classification_report, f1_score
print(classification_report(MERGED_BIOL, B_l))

## Combined Unigram + Bigram matching

In [127]:
import re

PARTICIPANTS = {
    
    "AGE": re.compile(
    r"\b("
    r"\d+\s*(years?|yrs?|yr|y)|"                 
    r"\d+\s*[-–]\s*\d+\s*(years?|yrs?|yr|y)|"    
    r"\d+\s*[±\+/-]\s*\d+\s*(years?|yrs?|yr|y)|" 
    r"aged?\s*\d+\s*(years?|yrs?|yr|y)?|"        
    r"adults?|children|infants?|elderly|"
    r"pediatric|neonates?|adolescents?|teenagers?"
    r")\b",
    re.IGNORECASE),
                
    "SEX": re.compile(
    r"\b("
    r"male|female|males?|females?|"
    r"men|women|"
    r"boys?|girls?|"
    r"both\s+sexes|"
    r"male\s+and\s+female|"
    r"men\s+and\s+women"
    r")\b",
    re.IGNORECASE),
    
    "SAMPLE SIZE": re.compile(
    r"\b("
    r"n\s*=\s*\d+|"                                      
    r"n\s*=\s*\d+\s*[-–]\s*\d+|"                            
    r"\d+\s+(patients?|subjects?|participants?)|"            
    r"\d+\s*[-–]\s*\d+\s+(patients?|subjects?|participants?)|" 
    r"a\s+total\s+of\s+\d+\s+(patients?|subjects?|participants?)|"  
    r"total\s+of\s+\d+\s+(patients?|subjects?|participants?)" 
    r")\b",
    re.IGNORECASE)
    }


INTERVENTION = {
    
    "SURGICAL": re.compile(
    r"\b("
    r"(surgical|operative)\s+(procedure|procedures|intervention|interventions|treatment|treatments)|"
    r"(surgery|surgeries|operation|operations)|"
    r"(procedure|procedures)"
    r")\b",
    re.IGNORECASE),
    
    "PHYSICAL": re.compile(
    r"\b("
    r"(physical)\s+(therapy|therapies|training|activity|activities|exercise|exercises|rehabilitation)|"
    r"(exercise|exercises|training|physiotherapy|rehabilitation|rehab|mobilization|mobilisation)|"
    r"(physical\s+therapy)"
    r")\b",
    re.IGNORECASE),
    
    "DRUG": re.compile(
    r"\b("
    r"(drug|drugs|medication|medications)\s+(therapy|treatment|regimen|administration)|"
    r"(pharmacological|pharmacologic)\s+(treatment|therapy|intervention)|"
    r"(dose|doses|dosage)\s+(adjustment|regimen|schedule|reduction|increase)|"
    r"(drug|drugs|medication|medications|pharmacotherapy|treatment|therapy|dose|doses|dosage)"
    r")\b",
    re.IGNORECASE),
    
    "EDUCATIONAL": re.compile(
    r"\b("
    r"(patient|health|self|group|caregiver)?\s*(education|educational)\s+(program|programme|intervention|session|training)|"
    r"(training|teaching|instruction)\s+(program|programme|session|intervention)|"
    r"(counseling|counselling)\s+(session|intervention|program|programme)|"
    r"(awareness)\s+(program|campaign)|"
    r"(education|educational|training|teaching|instruction|counseling|counselling|awareness|program|programme)"
    r")\b",
    re.IGNORECASE),
    
    "PSYCHOLOGICAL": re.compile(
    r"\b("
    r"(psychological|behavioral|behavioural|cognitive|mental)\s+(therapy|intervention|counseling|counselling|treatment|program|session)|"
    r"(psychotherapy|therapy|counseling|counselling)\s+(session|intervention|program|treatment)|"
    r"(psychological|behavioral|behavioural|cognitive|therapy|counseling|counselling|mental|psychotherapy)"
    r")\b",
    re.IGNORECASE)    
}



OUTCOMES = {

    "PHYSICAL": re.compile(
        r"\b("
        r"((longer|shorter|prolonged|extended|delayed|improved|worsened|increased|decreased|enhanced|declined|marked|significant|minor|slight|modest|severe|mild|moderate)\s+)?"
        r"(physical|functional|function)\s+(capacity|performance|recovery|improvement|mobility|movement|fitness|strength)|"
        r"((longer|shorter|prolonged|extended|delayed|improved|worsened|increased|decreased|enhanced|declined|marked|significant|minor|slight|modest|severe|mild|moderate)\s+)?"
        r"(mobility|movement|recovery|improvement|capacity|performance|strength|fitness)|"
        r"(physical|functional|function|mobility|movement|recovery|improvement|capacity|performance|strength|fitness)"
        r")\b",
        re.IGNORECASE
    ),

    "PAIN": re.compile(
        r"\b("
        r"((postoperative|post-op|chronic|acute|longer|shorter|prolonged|increased|decreased|severe|mild|moderate)\s+)?"
        r"(pain|ache|aches|soreness|discomfort|analgesia)\s+(relief|level|score|intensity|management|control|severity)|"
        r"((postoperative|post-op|chronic|acute|longer|shorter|prolonged|increased|decreased|severe|mild|moderate)\s+)?"
        r"(pain|ache|aches|soreness|discomfort|analgesia)|"
        r"(pain|ache|aches|soreness|discomfort|analgesia)"
        r")\b",
        re.IGNORECASE
    ),

    "MORTALITY": re.compile(
        r"\b("
        r"((postoperative|post-op|in-hospital|short-term|long-term|all-cause|longer|shorter|increased|decreased|higher|lower|significant|minor|major)\s+)?"
        r"(mortality|death|fatality|survival)\s*(rate|risk|probability|outcome|time|days)?|"
        r"(mortality|death|fatal|fatality|fatalities|survival)"
        r")\b",
        re.IGNORECASE
    ),

    "ADVERSE_EFFECTS": re.compile(
        r"\b("
        r"(side\s+effect|side\s+effects|adverse\s+effect|adverse\s+events|adverse\s+reaction|adverse\s+reactions|drug\s+related\s+adverse\s+effect)|"
        r"(complication|complications)\s+(rate|risk|incidence|occurrence)|"
        r"(bleeding|infection)\s+(rate|risk|incidence|occurrence)|"
        r"(adverse|toxicity|toxic|harm|risk|bleeding|infection|complication|complications)"
        r")\b",
        re.IGNORECASE
    ),

    "MENTAL": re.compile(
        r"\b("
        r"((longer|shorter|prolonged|extended|improved|worsened|increased|decreased|enhanced|declined|marked|significant|minor|slight|modest|severe|mild|moderate)\s+)?"
        r"(quality\s+of\s+life|QoL|well[-\s]?being|emotional\s+well[-\s]?being|emotional\s+state)|"
        r"((longer|shorter|prolonged|extended|improved|worsened|increased|decreased|enhanced|declined|marked|significant|minor|slight|modest|severe|mild|moderate)\s+)?"
        r"(mental|cognitive|mood|depression|depressive|anxiety|anxious|stress|stressed|psychological|emotion|emotional)\s+(level|score|state|status|outcome)|"
        r"(mental|cognitive|mood|depression|depressive|anxiety|anxious|stress|stressed|psychological|emotion|emotional)"
        r")\b",
        re.IGNORECASE
    )
}

In [128]:
def token_matching(token):
    token = token.lower().strip()

    for subtype, regex in OUTCOMES.items():
        if regex.search(token):
            return "OUT", subtype
        
    for subtype, regex in INTERVENTION.items():
        if regex.search(token):
            return "INT", subtype
        
    for subtype, regex in PARTICIPANTS.items():
        if regex.search(token):
            return "PART", subtype
        
    return "None", "None"

In [129]:
def bigram_tokens_matching(tok1, tok2):
    
    token = tok1.strip()+" "+tok2.strip()
    
    for subtype, regex in OUTCOMES.items():
        if regex.search(token):
            return "OUT", "OUT", subtype
        
    for subtype, regex in INTERVENTION.items():
        if regex.search(token):
            return "INT", "INT", subtype
        
    for subtype, regex in PARTICIPANTS.items():
        if regex.search(token):
            return "PART", "PART", subtype
        
    return "None", "None", "None"    

In [130]:
def entity_labled_tokens(doc_id, tok):
    tokens_list = [None] * len(tok)
    temp_dict = {"PARTICIPANTS":[], "INTERVENTION":[], "OUTCOMES":[]}
    
    i = 0
    while i < (len(tok)): 
        if i+1 < len(tok):
            entity1, entity2, subtype = bigram_tokens_matching(tok[i], tok[i+1])
            
            if  entity1 != "None":
                tokens_list[i]  = entity1
                tokens_list[i+1]= entity2
                
                if entity1 == "PART":
                    temp_dict["PARTICIPANTS"].append(tok[i].strip() + " " + tok[i+1].strip())  # to be updated if necessary for the subtype
            
                if entity1 == "INT":
                    temp_dict["INTERVENTION"].append(tok[i].strip() + " " + tok[i+1].strip())
                
                if entity1 == "OUT":
                    temp_dict["OUTCOMES"].append(tok[i].strip() + " " + tok[i+1].strip())
                    
                i += 2
                
                continue
            
        entity, subtype = token_matching(tok[i])
        tokens_list[i]  = entity
        
        if entity == "PART":
            temp_dict["PARTICIPANTS"].append(tok[i]) # to be updated if necessary for the subtype
                
        if entity == "INT":
            temp_dict["INTERVENTION"].append(tok[i])
               
        if entity == "OUT":
            temp_dict["OUTCOMES"].append(tok[i])
                
                
        i += 1
                
    return doc_id, tokens_list, temp_dict

In [131]:
def BIO_labels(token_list):
    prev = None
    bio_labels = []
    
    for i in token_list:
        if i  == "None":
            label = "O"
            
        elif i == "PART":
            if prev == "PART":
                label = "I" + "-PART"
            else:
                label = "B" + "-PART"
                
        elif i == "INT":
            if prev == "INT":
                label = "I" + "-INT"
            else:
                label = "B" + "-INT"
                
        elif i == "OUT":
            if prev == "OUT":
                label = "I" + "-OUT"
            else:
                label = "B" + "-OUT"
                
        bio_labels.append(label)
        prev = i
    
    return bio_labels

In [132]:
TOKENS = train_df["Tokens"].to_numpy()
DOC_ID = train_df["Doc_id"].to_numpy()
MERGED_BIOL = train_df["Merged BIO Labels"].to_numpy()


In [133]:
B_l = []
sub_l = []
for i in range(4457):
    doc_id = DOC_ID[i]
    tok = TOKENS[i]
    
    doc_id, tokens_list, temp_dict = entity_labled_tokens(doc_id, tok)
    BIO_Lab = BIO_labels(tokens_list)
    
    B_l.append(BIO_Lab)
    sub_l.append(temp_dict)

In [ ]:
from seqeval.metrics import classification_report, f1_score
print(classification_report(MERGED_BIOL, B_l))

              precision    recall  f1-score   support

         INT       0.05      0.03      0.03     29876
         OUT       0.09      0.03      0.05     31824
        PART       0.09      0.06      0.08     15245

   micro avg       0.07      0.04      0.05     76945
   macro avg       0.08      0.04      0.05     76945
weighted avg       0.07      0.04      0.05     76945



## spaCy-based linguistic processing and pattern matching

In [137]:
def token_matching(token):
    token = token.lower().strip()

    for subtype, regex in OUTCOMES.items():
        if regex.search(token):
            return "OUT", subtype
        
    for subtype, regex in INTERVENTION.items():
        if regex.search(token):
            return "INT", subtype
        
    for subtype, regex in PARTICIPANTS.items():
        if regex.search(token):
            return "PART", subtype
        
    return "None", "None"

In [138]:
def bigram_tokens_matching(tok1, tok2):
    
    token = tok1.strip()+" "+tok2.strip()
    
    for subtype, regex in OUTCOMES.items():
        if regex.search(token):
            return "OUT", "OUT", subtype
        
    for subtype, regex in INTERVENTION.items():
        if regex.search(token):
            return "INT", "INT", subtype
        
    for subtype, regex in PARTICIPANTS.items():
        if regex.search(token):
            return "PART", "PART", subtype
        
    return "None", "None", "None"    

In [139]:
import spacy

nlp = spacy.load("en_ner_bc5cdr_md")

def biomed_labels(text):
    
    doc = nlp(text)
    
    label_map = {"CHEMICAL": "INT", "DISEASE": "PART"}
    
    new_ents = []
    for ent in doc.ents:
        new_label = label_map.get(ent.label_, ent.label_) 
        new_ent = spacy.tokens.Span(doc, ent.start, ent.end, label=new_label)
        new_ents.append(new_ent)

    doc.ents = new_ents

    lab_dict = {}
    for ent in doc.ents:
        #print(ent.text, ent.label_)
        #lab_dict[ent.text] = ent.label_
        
        temp = ent.text.split()
        for i in temp:
            lab_dict[i.lower()] = ent.label_
    
        
    return lab_dict

In [140]:
def entity_labled_tokens(doc_id, doc, tok):
    
    tokens_list = [None] * len(tok)
    
    label_dict = biomed_labels(doc)
    
    temp_dict = {"PARTICIPANTS":[], "INTERVENTION":[], "OUTCOMES":[]}
    
    i = 0
    while i < (len(tok)):
        
        if tok[i].lower() in label_dict :
            entity =  label_dict[tok[i].lower()]
            tokens_list[i] = entity
            
            if entity == "PART":
                temp_dict["PARTICIPANTS"].append(tok[i]) # to be updated if necessary for the subtype
                    
            if entity == "INT":
                temp_dict["INTERVENTION"].append(tok[i])
                
            if entity == "OUT":
                temp_dict["OUTCOMES"].append(tok[i])
            
            i+=1
            
            continue
        
        elif i+1 < len(tok):
            entity1, entity2, subtype = bigram_tokens_matching(tok[i].lower(), tok[i+1].lower())
            
            if  entity1 != "None":
                tokens_list[i]  = entity1
                tokens_list[i+1]= entity2
                
                if entity1 == "PART":
                    temp_dict["PARTICIPANTS"].append(tok[i].strip() + " " + tok[i+1].strip())  # to be updated if necessary for the subtype
            
                if entity1 == "INT":
                    temp_dict["INTERVENTION"].append(tok[i].strip() + " " + tok[i+1].strip())
                
                if entity1 == "OUT":
                    temp_dict["OUTCOMES"].append(tok[i].strip() + " " + tok[i+1].strip())
                    
                i += 2
                
                continue
            
        entity, subtype = token_matching(tok[i].lower())
        tokens_list[i]  = entity
        
        if entity == "PART":
            temp_dict["PARTICIPANTS"].append(tok[i]) # to be updated if necessary for the subtype
                
        if entity == "INT":
            temp_dict["INTERVENTION"].append(tok[i])
               
        if entity == "OUT":
            temp_dict["OUTCOMES"].append(tok[i])
                
                
        i += 1
                
    return doc_id, tokens_list, temp_dict

In [141]:
def BIO_labels(token_list):
    prev = None
    bio_labels = []
    
    for i in token_list:
        if i  == "None":
            label = "O"
            
        elif i == "PART":
            if prev == "PART":
                label = "I" + "-PART"
            else:
                label = "B" + "-PART"
                
        elif i == "INT":
            if prev == "INT":
                label = "I" + "-INT"
            else:
                label = "B" + "-INT"
                
        elif i == "OUT":
            if prev == "OUT":
                label = "I" + "-OUT"
            else:
                label = "B" + "-OUT"
                
        bio_labels.append(label)
        prev = i
    
    return bio_labels

In [142]:
TOKENS = train_df["Tokens"].to_numpy()
DOC_ID = train_df["Doc_id"].to_numpy()
MERGED_BIOL = train_df["Merged BIO Labels"].to_numpy()
DOCS = train_df["Documents"]

In [143]:
B_l = []
sub_l = []
for i in range(4457):
    doc_id = DOC_ID[i]
    tok = TOKENS[i]
    doc = DOCS[i]
    
    doc_id, tokens_list, temp_dict = entity_labled_tokens(doc_id, doc, tok)
    BIO_Lab = BIO_labels(tokens_list)
    
    B_l.append(BIO_Lab)
    sub_l.append(temp_dict)

In [144]:
from seqeval.metrics import classification_report, f1_score
print(classification_report(MERGED_BIOL, B_l))

# Overall F1 score

              precision    recall  f1-score   support

         INT       0.21      0.32      0.25     29876
         OUT       0.08      0.03      0.04     31824
        PART       0.07      0.21      0.11     15245

   micro avg       0.14      0.18      0.15     76945
   macro avg       0.12      0.18      0.13     76945
weighted avg       0.13      0.18      0.14     76945

